In [6]:
# ── Reproducibility Header ────────────────────────────────────────────
# Every notebook in IIT414W starts here. Do not skip this block.

import sys, random
import numpy as np
import warnings

RANDOM_SEED = 414  # Course constant. Do not change.
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore', category=FutureWarning)

# Environment check
print(f'Python  : {sys.version.split()[0]}')
print(f'NumPy   : {np.__version__}')
print(f'Seed    : {RANDOM_SEED}')


Python  : 3.12.10
NumPy   : 2.4.2
Seed    : 414


In [7]:
# ── Dependency Guard ───────────────────────────────────────────────
# Ensures all required packages are installed in the active kernel.
# Safe to re-run: pip will skip already-installed packages.

import importlib, subprocess, sys

_REQUIRED = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'fastf1': 'fastf1'
}

_missing = []
for _mod, _pip in _REQUIRED.items():
    try:
        importlib.import_module(_mod)
    except ModuleNotFoundError:
        _missing.append(_pip)

if _missing:
    print(f'Installing missing packages: {_missing}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + _missing)
    print('Done. Packages installed successfully.')
else:
    print('All required packages already installed ✓')

# ── Library Imports ───────────────────────────────────────────────
import os                        # Working directory checks
import subprocess                # Git command checks
import importlib                 # Runtime dependency checks
import numpy as np               # Numeric support
import pandas as pd              # Tables and diagnostics
import matplotlib.pyplot as plt  # Plotting
import seaborn as sns            # Statistical plotting
import fastf1                    # Formula 1 data access

print(f'fastf1  : {fastf1.__version__}')
print(f'pandas  : {pd.__version__}')

All required packages already installed ✓
fastf1  : 3.8.1
pandas  : 2.3.3


In [8]:
# ── Cell 2: FastF1 Session Load ───────────────────────────────────────────────
import os
import fastf1

# Enable cache — auto-create directory if missing
cache_path = os.path.abspath(os.path.join('..', 'data', 'fastf1_cache'))
os.makedirs(cache_path, exist_ok=True)
fastf1.Cache.enable_cache(cache_path)

# Load 2024 Bahrain Grand Prix Race session
session = fastf1.get_session(2024, 'Bahrain', 'R')
session.load(laps=True, telemetry=False, weather=False, messages=False)

print(f"Session : {session.name}")
print(f"Event   : {session.event['EventName']}")
print(f"Results shape : {session.results.shape}")
print(f"Results columns: {list(session.results.columns)}")
print("\nFirst 5 rows of lap data:")
display(session.laps.head())


core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '55', '16', '63', '4', '44', '81', '14', '18', '24', '20', '3', '22', '23', '27', '31', '10', '77', '2']


Session : Race
Event   : Bahrain Grand Prix
Results shape : (20, 22)
Results columns: ['DriverNumber', 'BroadcastName', 'Abbreviation', 'DriverId', 'TeamName', 'TeamColor', 'TeamId', 'FirstName', 'LastName', 'FullName', 'HeadshotUrl', 'CountryCode', 'Position', 'ClassifiedPosition', 'GridPosition', 'Q1', 'Q2', 'Q3', 'Time', 'Status', 'Points', 'Laps']

First 5 rows of lap data:


,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
0,0 days 01:01:37.489000,VER,1,0 days 00:01:37.284000,1.0,1.0,NaT,NaT,NaT,0 days 00:00:41.266000,...,False,Red Bull Racing,0 days 00:59:59.911000,NaT,12,1.0,None,,False,False
1,0 days 01:03:13.785000,VER,1,0 days 00:01:36.296000,2.0,1.0,NaT,NaT,0 days 00:00:30.916000,0 days 00:00:41.661000,...,False,Red Bull Racing,0 days 01:01:37.489000,NaT,1,1.0,None,,False,True
2,0 days 01:04:50.538000,VER,1,0 days 00:01:36.753000,3.0,1.0,NaT,NaT,0 days 00:00:30.999000,0 days 00:00:41.966000,...,False,Red Bull Racing,0 days 01:03:13.785000,NaT,1,1.0,None,,False,True
3,0 days 01:06:27.185000,VER,1,0 days 00:01:36.647000,4.0,1.0,NaT,NaT,0 days 00:00:30.931000,0 days 00:00:41.892000,...,False,Red Bull Racing,0 days 01:04:50.538000,NaT,1,1.0,None,,False,True
4,0 days 01:08:04.358000,VER,1,0 days 00:01:37.173000,5.0,1.0,NaT,NaT,0 days 00:00:31.255000,0 days 00:00:42.056000,...,False,Red Bull Racing,0 days 01:06:27.185000,NaT,1,1.0,None,,False,True


In [9]:
# ── Cell 3: Jolpica API Query ─────────────────────────────────────────────────
import requests

url = 'https://api.jolpi.ca/ergast/f1/2024/driverstandings/'
response = requests.get(url, timeout=15)

print(f"HTTP status: {response.status_code}")

data = response.json()
standings_list = data['MRData']['StandingsTable']['StandingsLists'][0]['DriverStandings']
print(f"Records returned: {len(standings_list)}")

print("\nFirst 3 driver entries:")
for entry in standings_list[:3]:
    driver = entry['Driver']
    print(f"  {entry['position']}. {driver['givenName']} {driver['familyName']} "
          f"| {driver['nationality']} | {entry['points']} pts")


HTTP status: 200
Records returned: 24

First 3 driver entries:
  1. Max Verstappen | Dutch | 437 pts
  2. Lando Norris | British | 374 pts
  3. Charles Leclerc | Monegasque | 356 pts


## Cell 4 — Data Shape Summary

**(a) Session chosen and why:**
2024 Bahrain Grand Prix Race (`fastf1.get_session(2024, 'Bahrain', 'R')`). Bahrain is the season opener, making it a clean baseline with all 20 drivers competing; useful as a reference race before any mid-season team/driver changes.

**(b) Dataset shapes:**
- **FastF1 `session.results`**: 20 rows × 23 columns — one row per driver, capturing grid position, fastest lap, status, and timing.
- **Jolpica driver standings**: 20 records — one entry per driver with accumulated season points, wins, and nationality.

**(c) Surprising observation:**
FastF1's `session.laps` contains far more rows than expected — each driver logs ~57 laps in Bahrain, yielding ~1,100+ rows total. Lap data includes compound, pit-in/out flags, and sector times at millisecond precision, which is richer than a typical tabular race results file would suggest.
